In [2]:
import pandas as pd

df = pd.read_csv(
    "../notebooks/dense_gpt4o_results.csv"
)

print(
    df.columns.tolist()
)

['pipeline', 'retrieval_method', 'model', 'query', 'retrieved_count', 'retrieved_docs', 'context', 'context_length', 'generated_answer', 'response_length', 'latency_seconds', 'memory_mb']


In [3]:
# ============================================================
# LOAD ALL RETRIEVAL FILES
# ============================================================

import pandas as pd

dense_gpt4o = pd.read_csv("../notebooks/dense_gpt4o_results.csv").head(10)

hybrid_gpt4o = pd.read_csv("../notebooks/hybrid_gpt4o_results.csv").head(10)

fusion_gpt4o = pd.read_csv("../notebooks/fusion_gpt4o_results.csv").head(10)

dense_llama3 = pd.read_csv("../notebooks/dense_llama3_results.csv").head(10)

hybrid_llama3 = pd.read_csv("../notebooks/hybrid_llama3_results.csv").head(10)

fusion_llama3 = pd.read_csv("../notebooks/fusion_llama3_results.csv").head(10)

dense_mistral = pd.read_csv("../notebooks/dense_mistral_results.csv").head(10)

hybrid_mistral = pd.read_csv("../notebooks/hybrid_mistral_results.csv").head(10)

fusion_mistral = pd.read_csv("../notebooks/fusion_mistral_results.csv").head(10)

print("All Files Loaded")

All Files Loaded


In [4]:
# ============================================================
# COMBINE ALL PIPELINES
# ============================================================

all_dfs = [

    dense_gpt4o,
    hybrid_gpt4o,
    fusion_gpt4o,

    dense_llama3,
    hybrid_llama3,
    fusion_llama3,

    dense_mistral,
    hybrid_mistral,
    fusion_mistral

]

combined_df = pd.concat(
    all_dfs,
    ignore_index=True
)

print(
    combined_df.shape
)

combined_df.head()

(90, 12)


,pipeline,retrieval_method,model,query,retrieved_count,retrieved_docs,context,context_length,generated_answer,response_length,latency_seconds,memory_mb
0,Dense GPT4o,Dense,GPT4o,Harry Potter actor,5,['malfoy rupert grint ron weasley and director...,malfoy rupert grint ron weasley and director d...,2160,Daniel Radcliffe,16,1.065777,0.031250
1,Dense GPT4o,Dense,GPT4o,World Cup winner,5,['world cups including this summers tournament...,world cups including this summers tournament i...,2002,The context mentions several World Cup winners...,141,1.180831,0.250000
2,Dense GPT4o,Dense,GPT4o,US election,5,['president since 1964 going into the election...,president since 1964 going into the election n...,2494,The context provided discusses the 2008 U.S. p...,708,3.219878,0.109375
3,Dense GPT4o,Dense,GPT4o,climate change,5,['climate change let us know in the sound off ...,climate change let us know in the sound off bo...,2043,Climate change refers to the long-term alterat...,637,1.638212,0.000000
4,Dense GPT4o,Dense,GPT4o,Olympics,5,['one another thats what the olympics are abou...,one another thats what the olympics are about ...,2483,"The Olympics are about greatness, excellence, ...",587,2.047689,0.078125


In [5]:
# ============================================================
# CREATE MANUAL RELEVANCE SHEET
# ============================================================

relevance_df = combined_df[

    [

        "pipeline",

        "query",

        "retrieved_docs"

    ]

].copy()

relevance_df["relevant"] = 0

relevance_df.head()

,pipeline,query,retrieved_docs,relevant
0,Dense GPT4o,Harry Potter actor,['malfoy rupert grint ron weasley and director...,0
1,Dense GPT4o,World Cup winner,['world cups including this summers tournament...,0
2,Dense GPT4o,US election,['president since 1964 going into the election...,0
3,Dense GPT4o,climate change,['climate change let us know in the sound off ...,0
4,Dense GPT4o,Olympics,['one another thats what the olympics are abou...,0


In [6]:
# ============================================================
# EXPORT RELEVANCE FILE
# ============================================================

relevance_df.to_csv(

    "manual_relevance_labels.csv",

    index=False

)

print(
    "File Saved"
)

File Saved


In [7]:
# ============================================================
# LOAD LABELLED FILE
# ============================================================

labels_df = pd.read_csv(
    "manual_relevance_labels.csv"
)

labels_df.head()

,pipeline,query,retrieved_docs,relevant
0,Dense GPT4o,Harry Potter actor,['malfoy rupert grint ron weasley and director...,0
1,Dense GPT4o,World Cup winner,['world cups including this summers tournament...,0
2,Dense GPT4o,US election,['president since 1964 going into the election...,0
3,Dense GPT4o,climate change,['climate change let us know in the sound off ...,0
4,Dense GPT4o,Olympics,['one another thats what the olympics are abou...,0


In [8]:
# ============================================================
# PRECISION
# ============================================================

precision_results = []

for pipeline in labels_df["pipeline"].unique():

    temp = labels_df[
        labels_df["pipeline"] == pipeline
    ]

    precision = temp["relevant"].mean()

    precision_results.append({

        "pipeline":
        pipeline,

        "precision":
        precision

    })

precision_df = pd.DataFrame(
    precision_results
)

precision_df

,pipeline,precision
0,Dense GPT4o,0.0
1,Hybrid GPT4o,0.0
2,Fusion GPT4o,0.0
3,Dense Llama3,0.0
4,Hybrid Llama3,0.0
5,Fusion Llama3,0.0
6,Dense Mistral,0.0
7,Hybrid Mistral,0.0
8,Fusion Mistral,0.0


In [9]:
# ============================================================
# APPROXIMATE RECALL
# ============================================================

recall_results = []

for pipeline in labels_df["pipeline"].unique():

    temp = labels_df[
        labels_df["pipeline"] == pipeline
    ]

    recall = temp["relevant"].sum() / len(temp)

    recall_results.append({

        "pipeline":
        pipeline,

        "recall":
        recall

    })

recall_df = pd.DataFrame(
    recall_results
)

recall_df

,pipeline,recall
0,Dense GPT4o,0.0
1,Hybrid GPT4o,0.0
2,Fusion GPT4o,0.0
3,Dense Llama3,0.0
4,Hybrid Llama3,0.0
5,Fusion Llama3,0.0
6,Dense Mistral,0.0
7,Hybrid Mistral,0.0
8,Fusion Mistral,0.0


In [10]:
# ============================================================
# MRR
# ============================================================

mrr_results = []

for pipeline in labels_df["pipeline"].unique():

    temp = labels_df[
        labels_df["pipeline"] == pipeline
    ]

    if temp["relevant"].sum() > 0:

        first_rank = (

            temp.index[
                temp["relevant"] == 1
            ][0]

            + 1

        )

        mrr = 1 / first_rank

    else:

        mrr = 0

    mrr_results.append({

        "pipeline":
        pipeline,

        "MRR":
        mrr

    })

mrr_df = pd.DataFrame(
    mrr_results
)

mrr_df

,pipeline,MRR
0,Dense GPT4o,0
1,Hybrid GPT4o,0
2,Fusion GPT4o,0
3,Dense Llama3,0
4,Hybrid Llama3,0
5,Fusion Llama3,0
6,Dense Mistral,0
7,Hybrid Mistral,0
8,Fusion Mistral,0


In [12]:
# ============================================================
# FINAL RETRIEVAL METRICS
# ============================================================

final_df = precision_df.merge(

    recall_df,

    on="pipeline"

)

final_df = final_df.merge(

    mrr_df,

    on="pipeline"

)

final_df

,pipeline,precision,recall,MRR
0,Dense GPT4o,0.0,0.0,0
1,Hybrid GPT4o,0.0,0.0,0
2,Fusion GPT4o,0.0,0.0,0
3,Dense Llama3,0.0,0.0,0
4,Hybrid Llama3,0.0,0.0,0
5,Fusion Llama3,0.0,0.0,0
6,Dense Mistral,0.0,0.0,0
7,Hybrid Mistral,0.0,0.0,0
8,Fusion Mistral,0.0,0.0,0


In [13]:
# ============================================================
# SAVE RESULTS
# ============================================================

final_df.to_csv(

    "retrieval_metrics.csv",

    index=False

)

print(
    "Retrieval Metrics Saved"
)

Retrieval Metrics Saved
